# Chapter 7: Multi-Layer Perceptrons, Live Python Exercises

**ECUE 3.1.3: Neural Communication** &bull; CESI Bachelor AI (Year 2)

Reference notebook for **Chapter 7, Multi-Layer Perceptrons**, the three live demos of the chapter: activation functions and why a network needs them (part A), a feedforward network with matrices (part B), and learning a hidden layer by trial and error (part C).

The three demos are **fully written**: run them cell by cell, **predict** the output before each run, and change the values to see what happens. Around them, five **Your turn** exercises (a warm-up on the bias, activations by hand, two more networks, three stimuli from spike trains, tuning the blind search) have a **`# TODO`** cell for you to fill in. Keep the notebook as a reference for the rest of your studies. The answers to the questions, with all the outputs, are in `../notebooks-instructor/Chapter7_LiveExercises_SOLUTIONS.ipynb`.

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Sanity check:</b> run the cell below. If it prints a NumPy version with no error, your environment is ready. If not, go back to <code>Chapter1_Setup.ipynb</code>.
</div>

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)
print("Environment OK ✔")

---

## Warm-Up: Same Weights, Only the Bias Changes

*Paper first, then check in code.* A neuron with **equal weights** counts its active inputs; the bias sets the threshold.

1. Fix $w = (1, 1)$. Which gate do you get with $b = -2.5$, $-1.5$, $-0.5$ and $+0.5$?
2. Fix $w = (-1, -1)$. Which gate with $b = -0.5$, $0.5$, $1.5$ and $2.5$?
3. Three inputs, $w = (1, 1, 1)$. Which gate with $b = -0.5$, $-1.5$ and $-2.5$?

Then complete the cell below to print the truth table of every neuron, and check your answers.

In [ ]:
def mp_neuron(x, w, b):
    z = np.dot(w, x) + b       # weighted sum + bias (spatial summation)
    return 1 if z >= 0 else 0  # step function (all-or-nothing decision)


X_2 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
X_3 = np.array([[a, b, c] for a in (0, 1) for b in (0, 1) for c in (0, 1)])

# TODO: for w = [1, 1] and each bias of question 1, print the outputs on the 4 rows of X_2
# TODO: same for w = [-1, -1] (question 2), then w = [1, 1, 1] on the 8 rows of X_3 (question 3)

---

## Part A: Activations and the Collapse

### Step 1: four activation functions

`step`, `sigmoid`, `tanh` and `relu`, written so that they work on a whole NumPy array at once, and plotted for $z$ between $-5$ and $5$. **Predict**: which ones are smooth?

| | Step | Sigmoid | Tanh | ReLU |
|:---|:---:|:---:|:---:|:---:|
| $\varphi(z)$ | $1$ if $z \ge 0$, else $0$ | $\dfrac{1}{1 + e^{-z}}$ | $\tanh z$ | $\max(0, z)$ |

In [ ]:
import matplotlib.pyplot as plt


def step(z):
    return (z >= 0).astype(int)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def tanh(z):
    return np.tanh(z)


def relu(z):
    return np.maximum(0, z)


z = np.linspace(-5, 5, 201)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, f in zip(axes, [step, sigmoid, tanh, relu]):
    ax.plot(z, f(z))
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set_title(f.__name__)
plt.tight_layout()
plt.show()

### Your turn: activations by hand

*Paper first.* Fill in the table, then complete the cell below to check it.

| $z$ | $-2$ | $-0.5$ | $0$ | $0.5$ | $2$ |
|:---|:---:|:---:|:---:|:---:|:---:|
| step | | | | | |
| sigmoid | | | | | |
| tanh | | | | | |
| ReLU | | | | | |

*Help*: $e^{-2} \approx 0.14$, $e^{-0.5} \approx 0.61$, $e^{0.5} \approx 1.65$, $e^{2} \approx 7.39$.

Then: which activation looks most like the **firing-rate curve** of a real neuron (silent, then graded, then saturated)? Which one can output a **negative** value? Which one never saturates?

In [ ]:
z_values = np.array([-2, -0.5, 0, 0.5, 2])
# TODO: print each activation of part A on z_values, rounded to 2 decimals

### Step 2: two linear layers

Two layers **without** activation: $h = W_1 x + b_1$, then $y = W_2 h + b_2$. The cell below draws random weights: `W1` has shape `(3, 2)` (3 hidden neurons, 2 inputs), `W2` has shape `(1, 3)`.

`deep` is the output of the two layers for each of the 5 inputs in `X5`. `flat` is the output of the single layer $W' = W_2 W_1$, $b' = W_2 b_1 + b_2$. `np.allclose` compares them, up to rounding errors. (`@` is the matrix product in NumPy: `W1 @ x` computes all the weighted sums of the layer at once.)

**Predict before running**: True or False?

In [ ]:
rng = np.random.default_rng(0)
W1, b1 = rng.normal(size=(3, 2)), rng.normal(size=3)
W2, b2 = rng.normal(size=(1, 3)), rng.normal(size=1)
X5 = rng.normal(size=(5, 2))

deep = [W2 @ (W1 @ x + b1) + b2 for x in X5]

Wp, bp = W2 @ W1, W2 @ b1 + b2
flat = [Wp @ x + bp for x in X5]

print("two layers == one layer?", np.allclose(deep, flat))

### Step 3: add a ReLU between the layers

The same two layers, with a **ReLU** on the hidden layer: $y = W_2\,\text{relu}(W_1 x + b_1) + b_2$. **Predict before running**: still equal to `flat`?

In [ ]:
deep_relu = [W2 @ relu(W1 @ x + b1) + b2 for x in X5]

print("with ReLU, still one layer?", np.allclose(deep_relu, flat))

---

## Part B: A Feedforward Network in NumPy

A **layer** is several neurons reading the same inputs: $h = \varphi(W x + b)$, where row $j$ of $W$ holds the weights of neuron $j$.

### Step 1: one layer, then a whole network

- `layer(x, W, b, phi)` returns $\varphi(W x + b)$: the neuron of Chapter 3, for a whole layer at once.
- `forward(x, layers)`, where `layers` is a list of `(W, b, phi)`, applies them one after the other: the output of each layer is the input of the next. That's **communication** between layers.

In [ ]:
def layer(x, W, b, phi):
    return phi(W @ x + b)


def forward(x, layers):
    for W, b, phi in layers:
        x = layer(x, W, b, phi)    # output of this layer -> input of the next
    return x

### Step 2: XOR, with matrices

The XOR network of Chapter 6, as two layers: the hidden layer holds OR and NAND (one per row), the output layer is AND. **Predict**: what does `forward` return on the 4 rows?

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

xor_net = [
    (np.array([[1, 1], [-1, -1]]), np.array([-0.5, 1.5]), step),   # OR and NAND
    (np.array([[1, 1]]),           np.array([-1.5]),      step),   # AND
]

print("XOR:", [int(forward(x, xor_net)[0]) for x in X])

### Your turn: two more networks, with matrices

Chapter 6 built XOR as AND(OR, NAND). There are other ways.

1. **XOR, another way**: XOR is 1 when exactly one input is on, that is $(x_1 \text{ AND NOT } x_2)$ **OR** $(\text{NOT } x_1 \text{ AND } x_2)$. Both hidden gates are in the table of the 16 gates of Chapter 5, so each fits in one neuron. Find their weights, write the network `xor_net_2` as two layers, and check it with `forward`.
2. **XNOR**: in Chapter 6, it took a new hidden layer (AND, NOR). Now change **only the output layer** of `xor_net_2` so that the network computes XNOR (1 when both inputs are equal).
3. **Four gates in one layer**: write **one** layer `four_gates` (a $4 \times 2$ matrix and 4 biases) whose 4 neurons compute OR, AND, NAND and NOR at the same time. *Hint*: the warm-up.

In [ ]:
# TODO 1: xor_net_2 = [(W_hidden, b_hidden, step), (W_out, b_out, step)]
# TODO 2: xnor_net, same hidden layer, a new output layer
# TODO 3: four_gates = (W, b, step); print layer(x, *four_gates) for each row of X

### Step 3: more than two classes

Three neurons were recorded while one of **three** stimuli was shown (0, 1 or 2). Each stimulus drives mostly one neuron. An output layer has **one neuron per class**; the class whose neuron has the highest score wins: $\hat{c} = \arg\max_c s_c$.

With the hand-made output layer below (`W_out` = identity: each output neuron reads its preferred input), the next cell computes the scores of each trial and predicts with `np.argmax`. **Predict** the winning class of each trial before running.

In [ ]:
X3 = np.array([[40, 12, 10], [35, 15, 8], [10, 42, 12],
               [12, 38, 9], [9, 11, 45], [14, 10, 39]])
T3 = np.array([0, 0, 1, 1, 2, 2])

W_out = np.eye(3)
b_out = np.zeros(3)

In [ ]:
predictions = [int(np.argmax(W_out @ x + b_out)) for x in X3]

print("predictions:", predictions, "targets:", T3.tolist())

### Step 4: learn the output layer, the multiclass perceptron rule

`train_multiclass` learns `W_out` and `b_out` instead of writing them by hand. It starts from zeros; for each trial, it predicts with the argmax; **only when wrong**, it moves the right class toward $x$ and the wrong winner away from it:

$$w_{\text{true}} \leftarrow w_{\text{true}} + \eta\,x \qquad w_{\text{pred}} \leftarrow w_{\text{pred}} - \eta\,x$$

(same for the biases, with $1$ instead of $x$). This is exactly how the Project 2 decoder learns.

`train_multiclass` takes **any** feature matrix `X`, one row per trial: firing rates here, but just as well spike counts in time windows (Chapter 3, step 5), or the outputs of a hidden layer (Chapter 6: learn the last layer on the hidden representation). The rule does not care where the numbers come from.

In [ ]:
def train_multiclass(X, T, n_classes, eta=1.0, n_epochs=20):
    W, b = np.zeros((n_classes, X.shape[1])), np.zeros(n_classes)
    mistakes = []
    for epoch in range(n_epochs):
        errors = 0
        for x, t in zip(X, T):
            pred = int(np.argmax(W @ x + b))
            if pred != t:
                W[t] += eta * x
                b[t] += eta
                W[pred] -= eta * x
                b[pred] -= eta
                errors += 1
        mistakes.append(errors)
        if errors == 0:
            break
    return W, b, mistakes


W_l, b_l, mistakes = train_multiclass(X3, T3, n_classes=3)
print("mistakes per epoch:", mistakes)
print("predictions:", [int(np.argmax(W_l @ x + b_l)) for x in X3], "targets:", T3.tolist())

In [ ]:
# A 3 x 3 confusion matrix: rows = true class, columns = prediction
pred = [int(np.argmax(W_l @ x + b_l)) for x in X3]
confusion = np.zeros((3, 3), dtype=int)
for t, p in zip(T3, pred):
    confusion[t, p] += 1
print(confusion)

### Your turn: three stimuli, from spike trains

The same decoder as step 4, on recordings closer to Project 2. Four neurons, **three** stimuli. Each of the first three neurons prefers one stimulus; the fourth fires at 20 Hz whatever the stimulus.

| Neuron | 1 | 2 | 3 | 4 |
|:---:|:---:|:---:|:---:|:---:|
| Stimulus 0 *(Hz)* | 30 | 15 | 15 | 20 |
| Stimulus 1 *(Hz)* | 15 | 30 | 15 | 20 |
| Stimulus 2 *(Hz)* | 15 | 15 | 30 | 20 |

The cell below simulates the trials (spikes in 1 ms bins, as in Chapter 3 and Chapter 5): 20 training and 10 test trials of 1 s per stimulus.

1. Standardize the rates with the **training** statistics, then train `train_multiclass` for 50 epochs. Does it converge?
2. Compute the test accuracy and the $3 \times 3$ confusion matrix.
3. Print `W` rounded to 1 decimal. Read it **row by row** (one output neuron per class): which input neuron does each class rely on? And neuron 4?
4. *Bonus*: redo it with trials of 0.2 s. What happens to the confusion matrix?

In [ ]:
rng = np.random.default_rng(7)
dt = 0.001

rates = np.array([[30, 15, 15, 20],    # stimulus 0
                  [15, 30, 15, 20],    # stimulus 1
                  [15, 15, 30, 20]])   # stimulus 2


def record(r, n_trials, duration, rng):
    n_bins = int(duration / dt)
    X = np.zeros((n_trials, len(r)))
    for k in range(n_trials):
        spikes = (rng.random((len(r), n_bins)) < r[:, None] * dt).astype(int)
        X[k] = spikes.sum(axis=1) / duration    # firing rate in Hz
    return X


def dataset3(duration, rng, n_train=20, n_test=10):
    X_tr = np.vstack([record(r, n_train, duration, rng) for r in rates])
    T_tr = np.array([c for c in range(3) for _ in range(n_train)])
    X_te = np.vstack([record(r, n_test, duration, rng) for r in rates])
    T_te = np.array([c for c in range(3) for _ in range(n_test)])
    return X_tr, T_tr, X_te, T_te


X_tr, T_tr, X_te, T_te = dataset3(1.0, rng)
print("training set:", X_tr.shape, "| test set:", X_te.shape)


# TODO 1: standardize with mu and sigma of X_tr, then train_multiclass(..., n_classes=3, n_epochs=50)
# TODO 2: test accuracy and 3 x 3 confusion matrix
# TODO 3: print np.round(W, 1)

---

## Part C: Guided Demo, Let the Hidden Layer Learn

*Beyond the course: run the cells in order and read the outputs.*

So far, **we** picked every weight of the hidden layer. The XOR network has 9 numbers: 2 weights and 1 bias for each hidden neuron, and 2 weights and 1 bias for the output neuron. What happens if we reuse the **learning loop of the perceptron** (Chapter 5) on the whole network?

| Learning loop | Perceptron | Two-layer network, no hidden targets |
|:---|:---|:---|
| **1. Predict** | $y$ for one row | `network(x, p)`, with `p` = the 9 numbers |
| **2. Measure the error** | $t - y$, for this neuron | only at the output: `n_mistakes(p)` |
| **3. Correct** | $w_i \leftarrow w_i + \eta\,(t - y)\,x_i$: each weight, right direction | no target for $h_1, h_2$: nudge `p` at random, keep it if not worse |

### 1. Predict: the whole network as 9 numbers

| Neuron | Receives | Weights | Bias |
|:---|:---:|:---:|:---:|
| hidden 1 | $x_1, x_2$ | `p[0]`, `p[1]` | `p[2]` |
| hidden 2 | $x_1, x_2$ | `p[3]`, `p[4]` | `p[5]` |
| output | $h_1, h_2$ | `p[6]`, `p[7]` | `p[8]` |

`network(x, p)` makes the same three `mp_neuron` calls as `xor_network` in Chapter 6, but reads every weight and bias from `p`. Our hand-made network is just one particular `p`.

In [ ]:
def mp_neuron(x, w, b):
    z = np.dot(w, x) + b       # weighted sum + bias (spatial summation)
    return 1 if z >= 0 else 0  # step function (all-or-nothing decision)


def network(x, p):
    h = [mp_neuron(x, p[0:2], p[2]),   # hidden neuron 1
         mp_neuron(x, p[3:5], p[5])]   # hidden neuron 2
    return mp_neuron(h, p[6:8], p[8])  # output neuron


X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
T = np.array([0, 1, 1, 0])

#                  hidden 1 (OR)    hidden 2 (NAND)   output (AND)
p_hand = np.array([1, 1, -0.5,     -1, -1, 1.5,      1, 1, -1.5])
print("hand-made p ->", [network(x, p_hand) for x in X], "target:", T.tolist())

### 2. Measure the error: `n_mistakes`

Without targets for the hidden neurons, the only error we can measure is the network's **final** answer: `n_mistakes(p, X, T)` counts the rows of the XOR table where the output is wrong. `0` for the hand-made `p`; `2` for `p = 0` (every sum is 0, so every neuron fires).

What this error **cannot** tell us: *which* of the 9 numbers is to blame, and *in which direction* to move it.

In [ ]:
def n_mistakes(p, X, T):
    return sum(int(network(x, p) != t) for x, t in zip(X, T))


print("hand-made p ->", n_mistakes(p_hand, X, T), "mistake(s)")
print("p = 0       ->", n_mistakes(np.zeros(9), X, T), "mistake(s)")

### 3. Correct... blindly: one random nudge

Since the error gives no direction, we guess one: add a little random noise to **all 9 numbers** at once, measure again, and **keep** the new `p` if it makes no more mistakes (`m <= best`), otherwise throw it away. Five nudges from a random start:

In [ ]:
rng = np.random.default_rng(0)
p = rng.normal(size=9)
best = n_mistakes(p, X, T)
print("start : outputs", [network(x, p) for x in X], "->", best, "mistake(s)")

for i in range(1, 6):
    q = p + 0.5 * rng.normal(size=9)   # propose: nudge all 9 numbers
    m = n_mistakes(q, X, T)            # measure
    if m <= best:                      # keep it if it is not worse
        p, best = q, m
        decision = "keep"
    else:
        decision = "reject"
    print(f"try {i} : outputs {[network(x, q) for x in X]} -> {m} mistake(s) -> {decision}")

### Repeat: `random_search`

The same move in a loop, until 0 mistakes or `n_tries` tries. **Before running**: will blind guessing always find a solution?

In [ ]:
def random_search(X, T, n_tries=10000, step=0.5, seed=0):
    rng = np.random.default_rng(seed)
    p = rng.normal(size=9)
    best = n_mistakes(p, X, T)
    for i in range(1, n_tries + 1):
        q = p + step * rng.normal(size=9)   # random nudge of all 9 numbers
        m = n_mistakes(q, X, T)
        if m <= best:                         # keep it if it is not worse
            p, best = q, m
        if best == 0:
            break
    return p, best, i


results = [random_search(X, T, seed=s) for s in range(20)]
for s, (p, best, n) in enumerate(results):
    print(f"seed {s:2d}: {best} mistake(s) after {n:5d} tries")
print("solved:", sum(best == 0 for _, best, _ in results), "/ 20")

p0 = results[0][0]
print("seed 0 hidden representation:",
      [[mp_neuron(x, p0[0:2], p0[2]), mp_neuron(x, p0[3:5], p0[5])] for x in X])

### What to notice

1. Does the search always find a solution? What happens to the runs that fail?
2. Why can't a small random nudge get a stuck run out of trouble? *Hint: think about the step function.*
3. Did seed 0 find OR and NAND, like us?

### Your turn: tune the blind search

The blind search got stuck for 8 seeds out of 20. Two ideas to help it:

1. **Change the size of the nudge.** Run `random_search` on the 20 seeds with `step = 0.1`, `0.5` and `2.0`. For each, count the solved seeds and the median number of tries of the solved ones (`np.median`). Is there a "best" step?
2. **Restart when stuck.** Write `random_search_restart`: the same loop, but if `patience = 500` tries go by without a **strict** improvement, start again from a new random `p`. How many seeds out of 20 are solved now?
3. What does this tell you about learning without a direction?

In [ ]:
# TODO 1: for step in [0.1, 0.5, 2.0]: run random_search on seeds 0 to 19, count the solved ones, median tries


def random_search_restart(X, T, n_tries=10000, step=0.5, patience=500, seed=0):
    rng = np.random.default_rng(seed)
    p = rng.normal(size=9)
    best = n_mistakes(p, X, T)
    last_improvement = 0
    # TODO 2: the loop of random_search, plus: if i - last_improvement >= patience,
    #         restart from p = rng.normal(size=9)
    ...
    return p, best, i

---

## Feedforward, Backward Pass, Backpropagation

- **Biological neurons** communicate **one way**: dendrites, soma, axon, synapse.
- **Feedforward** (forward pass): input, hidden, output. The network **computes** its answer: that's what `forward` does.
- **Backward pass**: the **error** at the output is sent **back**, layer by layer, and tells each weight how much it contributed and in which direction to move.
- **Backpropagation** (Rumelhart, Hinton & Williams, 1986): the algorithm that computes the backward pass. It needs **smooth** activations, like the sigmoid of part A.

The guided demo had a forward pass, but no backward pass: it could only guess.